<a href="https://colab.research.google.com/github/GurneeshBudhiraja/ArangoDB-Hackathon/blob/main/ArangoDB_Hackathon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Testing Application using Arango DB and Synthea Dataset using Google Colab's T4 Runtime

In [1]:
# Required packages
!pip install nx-arangodb
!pip install arango-datasets
!nvidia-smi
!nvcc --version
!pip install nx-cugraph-cu12 --extra-index-url https://pypi.nvidia.com
!pip install google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.6/114.6 kB 4.7 MB/s eta 0:00:00
  Attempting uninstall: networkx
    Found existing installation: networkx 3.4.2
    Uninstalling networkx-3.4.2:
      Successfully uninstalled networkx-3.4.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.5.1+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.5.1+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-cupti-cu12 12.5.82 which

In [2]:
# Import required modules
from google.colab import userdata

import networkx as nx
import nx_arangodb as nxadb

from arango import ArangoClient
from arango_datasets import Datasets


# Gemini SDK Packages
from google import genai

[05:12:53 +0000] [INFO]: NetworkX-cuGraph is available.
INFO:nx_arangodb:NetworkX-cuGraph is available.
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:502: UserWarning: <built-in function any> is not a Python type (it may be an instance of an object), Pydantic will allow any object with no validation since we cannot even enforce that the input is an instance of the given type. To get rid of this error wrap the type with `pydantic.SkipValidation`.
  warn(


In [3]:
# Imports the secrets from the Colab notebook
ARANGO_HOST = userdata.get("ARANGO_HOST")
ARANGO_PASSWORD = userdata.get("ARANGO_PASSWORD")
ARANGO_USERNAME = userdata.get("ARANGO_USERNAME")
GEMINI_API = userdata.get("GEMINI_API_KEY")
FLASH_MODEL = "gemini-2.0-flash"


# Creates a db connection
db = ArangoClient(hosts=ARANGO_HOST).db(username=ARANGO_USERNAME, password=ARANGO_PASSWORD, verify=True)
gemini_client = genai.Client(
    api_key=GEMINI_API
)

In [4]:
# Creates the dataset connection using the db object
datasets = Datasets(db)

DATASET_NAME = "SYNTHEA_P100"

# Conditionally Loads the Synthea P100 dataset in Arango
if not db.has_graph(DATASET_NAME):
  datasets.load(dataset_name=DATASET_NAME)
else:
  print(f"{DATASET_NAME} is already in ArangoDB.")

Output()

SYNTHEA_P100 is already in ArangoDB.


In [5]:
# Connects with the Graph in ArangoDB
graph = None
if db.has_graph(DATASET_NAME):
  graph = nxadb.Graph(name=DATASET_NAME, db=db)
else:
  print("Graph does not exist in Arango DB")

print(graph)

[05:13:00 +0000] [INFO]: Graph 'SYNTHEA_P100' exists.
INFO:nx_arangodb:Graph 'SYNTHEA_P100' exists.
[05:13:01 +0000] [INFO]: Default node type set to 'allergies'
INFO:nx_arangodb:Default node type set to 'allergies'


Graph named 'SYNTHEA_P100' with 145514 nodes and 311701 edges


In [25]:
# Tools for the agent

# Tool to decide whether to use AQL, NetworkX or Hybrid to resolve the user query
def decide_method(user_query:str) -> str:
  """
  This function is responsible to decide whether to use AQL or NetworkX or Combined algorithms to further answer the user query.

  Args:
    user_query: The question asked by the user.

  Returns:
    Returns a string with the prefered method to use for the user_query
  """
  print("================= USING THE decide_method TOOL =================")
  system_instruction = f"""You are an expert in choosing the best method to query a graph database.
        You must choose one of the following methods:
        - aql: For direct database queries and relationship traversals.
        - networkX: For graph algorithm calculations.
        - hybrid: For combining AQL and NetworkX.

        Return ONLY a JSON object in the following format:

        {{
        "method": "method_name"
        }}

        Where 'method_name' is one of 'aql', 'networkX', or 'hybrid'. Do NOT return anything else.
        """
  response = gemini_client.models.generate_content(
      model=FLASH_MODEL,
      config={
          "system_instruction": system_instruction
      },
      contents=[user_query]
  )
  print(response.text)
  return response.text

# Tool to generate aql queries
def aql_generator(user_query: str) -> dict[str, str]:
  """
  This is used to generate the AQL queries using the user query

  Args:
    user_query: The query asked by the user

  Returns:
    Returns a dict with a key 'query' and AQL query which will be a string
  """
  print("================= USING THE aql_generator TOOL =================")
  return {"query":"This is a sample aql query"}


# Tool to generate networkX queries
def networkX_generator(user_query:str) -> dict[str, str]:
  """
  This is used to generate the NetworkX queries using the user query

  Args:
    user_query: The query asked by the user

  Returns:
    Returns a dict with a key 'query' which will be NetworkX query which will be a string
  """
  print("================= USING THE networkX_generator TOOL =================")
  return {"query":"This is a sample networkX query"}



In [40]:
tools = [decide_method,aql_generator,networkX_generator]
system_instructions = """You are the smart agent whose main job is to use the required tools to help answer the user query when the data that is stored is Synthea_P100. These are the tools you have access to:

For now your only job is to call the required tools to let the user know what method/approach would be best to go for the given user_query. Also, if a tool is available to generate a query for the method selected, use that tool as well and then return the final response to the user.
Please note there would be instances where the response of one tool would heavily depend on the response of the previous tool. So make sure to run the tools in the proper order and also you may need to run more than one tool to reach to the final answer.
"""

In [41]:

config = {
    'tools': tools,
    'system_instruction': system_instructions,
}


In [42]:
user_input = input("Enter your query: ").strip()
agent_response = gemini_client.models.generate_content(
    model=FLASH_MODEL,
    config=config,
    contents=[user_input]
)
print(f"Agent response is:")
print(agent_response.text)


Enter your query: What are the most common conditions among patients who are in the largest community of patients with similar allergies?
================= USING THE decide_method TOOL =================
{"method": "hybrid"}

================= USING THE networkX_generator TOOL =================
================= USING THE aql_generator TOOL =================
Agent response is:
The approach to answer the user query would be a hybrid approach, meaning a combination of both AQL and NetworkX. First, a NetworkX query will be generated to find patients with similar allergies, and then an AQL query will be generated to find the most common conditions among those patients.

Here are the generated queries:
NetworkX query:
```
This is a sample networkX query
```
AQL query:
```
This is a sample aql query
```
